# Three shapes of learning problem

MichAl Academy, lesson 2.1.

Run each cell with **Shift+Enter**.

Three shapes, and the thing that decides which one you have is what you already
possess rather than what you would like to build. The first two shapes use one
small table of flower measurements. The third one uses no table at all, which is
the point of it.


## 1. The table

The iris dataset: 150 flowers, four measurements each, and the species. It ships
inside scikit-learn, so there is nothing to download.

It has been the standard teaching set since 1936 because it is small enough to
read and awkward enough to be interesting.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame

print(df.shape)
print(iris.target_names)
df.head()


In [ ]:
# 50 of each species, and the answer lives in the `target` column.
print(df.target.value_counts().sort_index().to_string())


Notice where that `target` column came from. Somebody went out with calipers,
measured 150 flowers and wrote down what each one was. Every label in every
supervised dataset you will ever use exists because a person or another machine
produced it, and that is the expensive part.


## 2. Supervised: the answer is in the table

You have four numbers per flower and the species for all 150. So ask the direct
question: given the measurements, which species is this?


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

X = iris.data.to_numpy()
y = iris.target.to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

scaler = StandardScaler().fit(X_train)
model = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train)
pred = model.predict(scaler.transform(X_test))

print("accuracy", round(accuracy_score(y_test, pred), 3))
print()
print(pd.DataFrame(
    confusion_matrix(y_test, pred),
    index=[f"was {n}" for n in iris.target_names],
    columns=[f"said {n}" for n in iris.target_names],
))


Forty-four of the forty-five held-back flowers named correctly, and the one
mistake is a virginica called a versicolor. Those two species overlap, and every
method in this track will make its mistakes in the same place.

Only the label made that question askable. Take the `target` column away and
there is nothing to be right or wrong about.


## 3. Unsupervised: take the answer away

Now hide the species. All you have is four measurements per flower and a
suspicion that there is structure in there. Ask for the flowers to be grouped.


In [ ]:
from sklearn.cluster import KMeans

Z = StandardScaler().fit_transform(X)          # cluster on all 150, no labels used
species = [iris.target_names[i] for i in y]

df["k2"] = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(Z)
print(pd.crosstab(df.k2, species))


Asked for two groups, it put setosa on one side and the other hundred flowers on
the other. That is the largest split in the data and it is entirely real. It is
also not the three species you were after.

Fair objection: you asked for two groups when there are three. So tell it the
number.


In [ ]:
df["k3"] = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(Z)
print(pd.crosstab(df.k3, species))


Setosa comes out exactly. Twenty-five of the remaining hundred flowers land in
the wrong group, even though the number of groups was handed over.

Nothing malfunctioned. K-means minimises distance inside each cluster, and the
boundary between versicolor and virginica is not where the widest gap in the
measurements is. It answered the question it was asked, correctly, and that
question was never quite yours.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True, sharey=True)
petal_l, petal_w = df["petal length (cm)"], df["petal width (cm)"]
palette = np.array(["#00798c", "#d1495b", "#edae49"])

for ax, colour_by, title in [
    (axes[0], y,          "the three species"),
    (axes[1], df.k2,      "k-means, two groups"),
    (axes[2], df.k3,      "k-means, three groups"),
]:
    ax.scatter(petal_l, petal_w, c=palette[np.asarray(colour_by)], s=18)
    ax.set_title(title)
    ax.set_xlabel("petal length (cm)")

axes[0].set_ylabel("petal width (cm)")
plt.tight_layout()
plt.show()


Same 150 points in all three panels. The data never moved. What changed is
whether anything told the method where the answer was.


## 4. Reinforcement: no table, just a score

The third shape does not start from a dataset. Five machines, each paying out at
its own unknown rate. You get a thousand pulls. Win as much as you can.

There is no correct answer to copy, only a payout that arrives after you act.
That is the reinforcement learning setting, stripped to the smallest thing that
still counts as one.


In [ ]:
PAYOUT = [0.20, 0.35, 0.55, 0.40, 0.30]     # unknown to the agent


def play(rounds=1000, eps=0.10, seed=0):
    """Epsilon-greedy. Ten percent of pulls explore at random, the rest go to
    whichever machine currently looks best."""
    r = np.random.default_rng(seed)
    n = len(PAYOUT)
    pulls = np.zeros(n, int)
    value = np.zeros(n)                      # running average payout per machine

    for _ in range(rounds):
        a = int(r.integers(n)) if r.random() < eps else int(np.argmax(value))
        won = r.random() < PAYOUT[a]
        pulls[a] += 1
        value[a] += (won - value[a]) / pulls[a]

    return pulls, value


pulls, value = play()
print(f"{'machine':>8} {'true':>6} {'learned':>8} {'pulls':>6}")
for i, (t, v, p) in enumerate(zip(PAYOUT, value, pulls), start=1):
    print(f"{i:>8} {t:>6.2f} {v:>8.3f} {p:>6}")
print()
print("settled on machine", int(np.argmax(value)) + 1)


It found machine 3 without ever being told the payouts.

The pull counts are the more interesting output. Machines 3 and 4 took 929 of
the thousand pulls between them, and the three worst machines shared the
remaining 71. Machine 4 pays 0.40 against machine 3's 0.55, so it was the one
worth arguing about, and that is where the money went. Every pull spent
measuring a machine you have already written off is a pull you do not get to
spend winning, so an agent explores as little as it can get away with.


## 5. What that cost in trials

Same agent, different numbers of pulls, three hundred fresh runs each. How often
does it end up on the best machine?


In [ ]:
best = int(np.argmax(PAYOUT))

for rounds in (30, 100, 300, 1000, 3000):
    found = sum(int(np.argmax(play(rounds, seed=s)[1])) == best for s in range(300))
    print(f"{rounds:>5} pulls   best machine found in {found / 300:6.1%} of runs")


With thirty pulls it is barely better than guessing. It needs hundreds of
attempts at a five-way choice with a single number to learn.

That is the constraint on the whole approach, not a weakness of this small
example. Goecks, reviewing the methods that try to make reinforcement learning
less data-hungry, notes that current end-to-end approaches "still require
thousands or millions of data samples to converge to a satisfactory policy".
Reinforcement learning therefore lives where trials are cheap and harmless:
games, simulators, recommendation.

You will meet the same machinery again in lesson 2.8, where the choice is an
alerting threshold and the payout is the cost of each kind of mistake, and in
lesson 4.4, where the reward is a human saying which of two answers was better.


## What to take from this

| You have | You want | Shape |
|---|---|---|
| A label on every row | To predict that label | Supervised |
| No labels | To know what structure is in there | Unsupervised |
| An action, and a score that arrives afterwards | The action with the best score | Reinforcement |

The question to ask in front of a real dataset is not which algorithm to use. It
is: **does an answer exist for every row, and did anybody check it?** Everything
else follows from that.
